In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 3.8 MB/s eta 0:00:00


In [3]:
# Import Libraries
import requests
import pandas as pd
import json
from typing import List, Dict, Any
import anthropic

In [4]:
# Seed Input Samples
# SEED_INPUT = "diabetes mellitus type 2"
# SEED_INPUT = "breast cancer"
# SEED_INPUT = "breast cancer by Pfizer"
SEED_INPUT = "breast cancer durvalumab by AstraZeneca"
# SEED_INPUT = "AstraZeneca colorectal cancer"
# SEED_INPUT = 'breast cancer Bristol Myers Squibb'
# SEED_INPUT = 'COVID-19 vaccine trials'
# SEED_INPUT = 'lung cancer by AstraZeneca'

# API Configuration: Using Clinical Trials Database
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
MAX_RESULTS = 20

In [ ]:
import os
import re

CLAUDE_API_KEY = "ANTHROPIC_API_KEY"

if not CLAUDE_API_KEY:
    print("Warning: CLAUDE_API_KEY not set!")

# Function to process seed input with LLM
def process_seed_input_with_llm(seed_input: str) -> Dict[str, str]:
    """
    Use LLM to understand and parse the natural language seed input.
    Extracts key components like disease, intervention, sponsor, etc.
    """
    if not CLAUDE_API_KEY:
        return {"condition": seed_input, "intervention": "", "sponsor": "", "target": "", "keywords": ""}

    try:
        client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

        prompt = f"""
        Analyze the following natural language query for clinical trial search: "{seed_input}"

        Extract the following components if present:
        - Disease/Condition: The medical condition or disease mentioned
        - Intervention/Drug: Any specific treatment, drug, or intervention
        - Sponsor/Company: Any mentioned company or sponsor
        - Target: Any molecular target if specified
        - Other keywords: Any other relevant terms

        Return ONLY a valid JSON object with keys: condition, intervention, sponsor, target, keywords
        Do not include any other text, explanations, or formatting. Just the JSON.
        Example: {{"condition": "breast cancer", "intervention": "", "sponsor": "", "target": "", "keywords": ""}}
        """

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=200,
            temperature=0.1,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        result = response.content[0].text.strip()

        # Extract JSON if wrapped in code blocks
        json_match = re.search(r'```json\s*(\{.*?\})\s*```', result, re.DOTALL)
        if json_match:
            result = json_match.group(1).strip()

        # Parse JSON
        try:
            parsed = json.loads(result)
            return parsed
        except json.JSONDecodeError as je:
            print(f"JSON parsing failed: {je}. Response was: '{result}'.")
            return {"condition": seed_input, "intervention": "", "sponsor": "", "target": "", "keywords": ""}

    except Exception as e:
        print(f"LLM processing failed: {e}.")
        return {"condition": seed_input, "intervention": "", "sponsor": "", "target": "", "keywords": ""}

# Process the seed input
PARSED_SEED = process_seed_input_with_llm(SEED_INPUT)
print("Parsed seed input:")
print(PARSED_SEED)

Parsed seed input:
{'condition': 'breast cancer', 'intervention': 'durvalumab', 'sponsor': 'AstraZeneca', 'target': '', 'keywords': ''}


In [6]:
def collect_trial_data(parsed_seed: Dict[str, str], max_results: int = 20) -> pd.DataFrame:
    """
    Collect and structure clinical trial data based on parsed seed input.
    Searches for trials and fetches details in one integrated process.
    """
    seed_description = f"condition: {parsed_seed.get('condition', '')}, intervention: {parsed_seed.get('intervention', '')}, sponsor: {parsed_seed.get('sponsor', '')}"
    print(f"Searching for clinical trials related to: {seed_description}...")

    # Search parameters
    params = {
        "format": "json",
        "countTotal": "true",
        "pageSize": max_results
    }

    # Use specific API parameters for each component
    if parsed_seed.get("condition"):
        params["query.cond"] = parsed_seed["condition"]
    if parsed_seed.get("intervention"):
        params["query.intr"] = parsed_seed["intervention"]
    if parsed_seed.get("sponsor"):
        params["query.spons"] = parsed_seed["sponsor"]

    # Use general term for targets/keywords
    additional_terms = []
    if parsed_seed.get("target"):
        additional_terms.append(parsed_seed["target"])
    if parsed_seed.get("keywords"):
        additional_terms.append(parsed_seed["keywords"])
    if additional_terms:
        params["query.term"] = " ".join(additional_terms)

    print(f"Search parameters: {params}")

    # Search for trials
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    print("response",response.json())
    data = response.json()
    studies = data.get("studies", [])
    print(f"Found {len(studies)} trials.")

    # Fetch details for each trial directly
    trial_data = []
    for i, study in enumerate(studies):
        nct_id = study.get("protocolSection", {}).get("identificationModule", {}).get("nctId")
        if nct_id:
            print(f"Fetching details for {nct_id} ({i+1}/{len(studies)})...")
            # Double-check sponsor filtering
            sponsor = study.get("protocolSection", {}).get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name", "")

            if parsed_seed.get("sponsor"):
              if parsed_seed["sponsor"].lower() not in sponsor.lower():
                continue  # Skip trials that don't match the sponsor
            try:
                # Get full details for this trial
                detail_params = {"format": "json"}
                detail_url = f"{BASE_URL}/{nct_id}"
                detail_response = requests.get(detail_url, params=detail_params)
                detail_response.raise_for_status()

                detail_data = detail_response.json()
                full_study = detail_data.get("protocolSection", {})

                # Extract key information
                details = {
                    "nct_id": nct_id,
                    "title": full_study.get("identificationModule", {}).get("briefTitle", ""),
                    "conditions": ", ".join(full_study.get("conditionsModule", {}).get("conditions", [])),
                    "interventions": ", ".join([
                        f"{intv.get('name', '')} ({intv.get('type', '')})"
                        for intv in full_study.get("armsInterventionsModule", {}).get("interventions", [])
                    ]),
                    "sponsor": sponsor,
                    # "sponsor": full_study.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name", ""),
                    "status": full_study.get("statusModule", {}).get("overallStatus", ""),
                    "brief_summary": full_study.get("descriptionModule", {}).get("briefSummary", ""),
                    "start_date": full_study.get("statusModule", {}).get("startDateStruct", {}).get("date", ""),
                    "completion_date": full_study.get("statusModule", {}).get("completionDateStruct", {}).get("date", "")
                }

                trial_data.append(details)
            except Exception as e:
                print(f"Error fetching {nct_id}: {e}")
    df = pd.DataFrame(trial_data)
    return df

In [7]:
trials_df = collect_trial_data(PARSED_SEED, MAX_RESULTS)

Searching for clinical trials related to: condition: breast cancer, intervention: durvalumab, sponsor: AstraZeneca...
Search parameters: {'format': 'json', 'countTotal': 'true', 'pageSize': 20, 'query.cond': 'breast cancer', 'query.intr': 'durvalumab', 'query.spons': 'AstraZeneca'}
response {'totalCount': 42, 'studies': [{'protocolSection': {'identificationModule': {'nctId': 'NCT02628132', 'orgStudyIdInfo': {'id': '2151-169'}, 'secondaryIdInfos': [{'id': 'ESR-14-10649', 'type': 'OTHER_GRANT', 'domain': 'AstraZeneca'}], 'organization': {'fullName': 'King Faisal Specialist Hospital & Research Center', 'class': 'OTHER'}, 'briefTitle': 'Study of Safety and Efficacy of Durvalumab in Combination With Paclitaxel in Metastatic Triple Negative Breast Cancer Patients', 'officialTitle': 'Study of the Safety, Tolerability and Efficacy of the Investigational Anti PD-L1 Monoclonal Antibody Durvalumab in Combination With Paclitaxel in Patients With Metastatic Triple Negative PD-L1 Positive Breast Can

In [8]:
trials_df

,nct_id,title,conditions,interventions,sponsor,status,brief_summary,start_date,completion_date
0,NCT02734004,A Phase I/II Study of MEDI4736 in Combination ...,"Ovarian, Breast, SCLC, Gastric Cancers","Olaparib (DRUG), MEDI4736 (DRUG), Bevacizumab ...",AstraZeneca,ACTIVE_NOT_RECRUITING,The purpose of this study is to look at the ef...,2016-03-17,2026-09-17
1,NCT02264678,Ascending Doses of Ceralasertib in Combination...,"Adv Solid Malig - H&N SCC, ATM Pro / Def NSCLC...","Administration of ceralasertib (DRUG), Adminis...",AstraZeneca,ACTIVE_NOT_RECRUITING,"This is a modular, phase I/ phase 1 b, open-la...",2014-10-31,2025-12-31
2,NCT06103864,A Phase III Study of Dato-DXd With or Without ...,Breast Cancer,"Dato-DXd (DRUG), Durvalumab (DRUG), Paclitaxel...",AstraZeneca,RECRUITING,"This is a Phase III, randomised, open-label, 3...",2023-11-23,2029-04-30
3,NCT02586987,"A Study to Assess the Safety, Tolerability and...","Lung Cancer, Melanoma, Head and Neck Carcinoma...","Selumetinib (DRUG), MEDI4736 (DRUG), Tremelimu...",AstraZeneca,COMPLETED,"This is a Phase I, open-label, multi-centre, d...",2015-12-28,2019-09-20
4,NCT04556773,A Phase 1b Study of T-DXd Combinations in HER2...,Metastatic Breast Cancer,"Trastuzumab deruxtecan (DRUG), Durvalumab (DRU...",AstraZeneca,ACTIVE_NOT_RECRUITING,"DESTINY-Breast 08 will investigate the safety,...",2020-12-17,2026-06-16
5,NCT02658214,Durvalumab and Tremelimumab in Combination Wit...,"Small Cell Lung Carcinoma, Carcinoma, Squamous...","paclitaxel + carboplatin (DRUG), carboplatin +...",AstraZeneca,COMPLETED,Durvalumab and Tremelimumab in combination wit...,2016-04-28,2019-11-14
6,NCT04538742,A Phase 1b/2 Study of T-DXd Combinations in HE...,Metastatic Breast Cancer,"Trastuzumab deruxtecan (DRUG), Durvalumab (DRU...",AstraZeneca,ACTIVE_NOT_RECRUITING,"DESTINY-Breast07 will investigate the safety, ...",2020-12-28,2030-01-31


In [9]:
def categorize_intervention(intervention_text: str) -> Dict[str, str]:
    """
    Use AI to categorize intervention type and extract specific names/agents.
    """
    if not CLAUDE_API_KEY or not intervention_text:
        return {"type": "Unknown", "drugs": []}

    try:
        import anthropic
        client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

        prompt = f"""
        Analyze this intervention description and categorize it.

        Return JSON with:
        - "type": One of "Drug", "Biological", "Device", "Procedure", "Diagnostic Test", "Other"
        - "drugs": Array of specific names mentioned (drugs, biological agents, devices, tests, procedures)

        Examples:
        - "Drug: Aspirin (DRUG)" → {{"type": "Drug", "drugs": ["Aspirin"]}}
        - "Biological: Pembrolizumab (BIOLOGICAL)" → {{"type": "Biological", "drugs": ["Pembrolizumab"]}}
        - "Diagnostic Test: BRCA1 Gene Test" → {{"type": "Diagnostic Test", "drugs": ["BRCA1 Gene Test"]}}
        - "Device: Pacemaker Implant" → {{"type": "Device", "drugs": ["Pacemaker"]}}

        Intervention: "{intervention_text}"

        Extract ALL specific names, not just traditional drugs.
        """

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=200,
            temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )

        result = response.content[0].text.strip()
        parsed = None

        # Extract from markdown code blocks
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', result, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group(1).strip())
            except json.JSONDecodeError:
                pass

        # Find JSON-like content
        if not parsed:
            json_start = result.find('{')
            json_end = result.rfind('}') + 1
            if json_start != -1 and json_end > json_start:
                potential_json = result[json_start:json_end]
                try:
                    parsed = json.loads(potential_json)
                except json.JSONDecodeError:
                    pass

        # Try entire response
        if not parsed:
            try:
                parsed = json.loads(result)
            except json.JSONDecodeError:
                pass

        # Manual extraction
        if not parsed:
            intervention_type = "Unknown"
            extracted_names = []

            # Determine type from text patterns
            text_lower = intervention_text.lower()
            if any(word in text_lower for word in ['drug', 'medication', 'tablet', 'oral']):
                intervention_type = "Drug"
            elif any(word in text_lower for word in ['biological', 'antibody', 'vaccine', 'cell therapy', 'monoclonal']):
                intervention_type = "Biological"
            elif any(word in text_lower for word in ['device', 'implant', 'stent', 'catheter']):
                intervention_type = "Device"
            elif any(word in text_lower for word in ['procedure', 'surgery', 'therapy', 'radiation']):
                intervention_type = "Procedure"
            elif any(word in text_lower for word in ['test', 'assay', 'diagnostic', 'screening', 'biomarker']):
                intervention_type = "Diagnostic Test"

            # Extract names using multiple patterns
            # Pattern 1: Parentheses content (DRUG), (BIOLOGICAL), etc.
            paren_matches = re.findall(r'\(([^)]+)\)', intervention_text)
            for match in paren_matches:
                if match.upper() in ['DRUG', 'BIOLOGICAL', 'DEVICE', 'PROCEDURE', 'DIAGNOSTIC TEST']:
                    continue  # Skip the type indicators
                extracted_names.append(match.strip())

            # Pattern 2: Colon-separated values
            if ':' in intervention_text:
                parts = intervention_text.split(':')
                if len(parts) > 1:
                    name_part = parts[1].strip()
                    # Remove type indicators
                    name_part = re.sub(r'\s*\([^)]+\)\s*$', '', name_part)
                    if name_part and len(name_part.split()) <= 5:  # Reasonable length limit
                        extracted_names.append(name_part)

            # Pattern 3: Capitalized words (potential drug/device names)
            capitalized_words = re.findall(r'\b[A-Z][a-zA-Z0-9\-]+(?:\s+[A-Z][a-zA-Z0-9\-]+)*\b', intervention_text)
            for word in capitalized_words:
                if len(word.split()) <= 3 and word not in ['The', 'And', 'Or', 'For', 'With', 'Drug', 'Biological', 'Device']:
                    extracted_names.append(word)

            # Remove duplicates and limit
            extracted_names = list(set(extracted_names))[:5]  # Max 5 names

            parsed = {"type": intervention_type, "drugs": extracted_names}

        # Validate the result
        if not isinstance(parsed, dict) or 'type' not in parsed or 'drugs' not in parsed:
            parsed = {"type": "Unknown", "drugs": []}

        if not isinstance(parsed['drugs'], list):
            parsed['drugs'] = []

        return parsed

    except Exception as e:
        print(f"Intervention categorization failed: {e}")
        return {"type": "Unknown", "drugs": []}

In [10]:
def generate_trial_summary(trial_data: Dict[str, Any]) -> str:
    """
    Use AI to generate a structured summary of the trial.
    """
    if not CLAUDE_API_KEY:
        return trial_data.get('brief_summary', '')

    try:
        import anthropic
        client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

        prompt = f"""
        Generate a concise, structured summary of this clinical trial in 2-3 sentences.
        Focus on: what condition it treats, what intervention is tested, current status, and key outcomes if available.

        Trial Data:
        - Title: {trial_data.get('title', '')}
        - Condition: {trial_data.get('conditions', '')}
        - Intervention: {trial_data.get('interventions', '')}
        - Sponsor: {trial_data.get('sponsor', '')}
        - Status: {trial_data.get('status', '')}
        - Start Date: {trial_data.get('start_date', '')}
        - Completion Date: {trial_data.get('completion_date', '')}
        - Summary: {trial_data.get('brief_summary', '')}
        """

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=150,
            temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )

        return response.content[0].text.strip()

    except Exception as e:
        print(f"Summary generation failed: {e}")
        return trial_data.get('brief_summary', '')

def trial_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply AI processing to enrich the dataset with intervention categorization and AI summaries.
    """
    if df.empty:
        return df

    final_df = df.copy()

    # Add new columns
    final_df['intervention_category'] = None
    final_df['extracted_drugs'] = None
    final_df['ai_summary'] = None

    for idx, row in final_df.iterrows():
        print(f"Processing trial {idx+1}/{len(final_df)}: {row['nct_id']}")

        # Categorize intervention
        intervention_info = categorize_intervention(row.get('interventions', ''))
        final_df.at[idx, 'intervention_category'] = intervention_info.get('type', 'Unknown')
        final_df.at[idx, 'extracted_drugs'] = intervention_info.get('drugs', [])

        # Generate AI summary
        ai_summary = generate_trial_summary(row.to_dict())
        final_df.at[idx, 'ai_summary'] = ai_summary

    return final_df

In [11]:
# Display the results
if not trials_df.empty:
    print(f"Collected data for {len(trials_df)} clinical trials:")

    enriched_trials_df = trial_dataset(trials_df)

    display(enriched_trials_df.head())

    # Summary Statistics
    print("\nSummary:")
    print(f"- Status distribution: {enriched_trials_df['status'].value_counts().to_dict()}")
    print(f"- Intervention categories: {enriched_trials_df['intervention_category'].value_counts().to_dict()}")

    # Save dataset to CSV
    enriched_trials_df.to_csv("clinical_trials.csv", index=False)
    print("\nData saved to 'clinical_trials.csv'")
else:
    print("No trial data collected.")

Collected data for 7 clinical trials:
Processing trial 1/7: NCT02734004
Processing trial 2/7: NCT02264678
Processing trial 3/7: NCT06103864
Processing trial 4/7: NCT02586987
Processing trial 5/7: NCT04556773
Processing trial 6/7: NCT02658214
Processing trial 7/7: NCT04538742


,nct_id,title,conditions,interventions,sponsor,status,brief_summary,start_date,completion_date,intervention_category,extracted_drugs,ai_summary
0,NCT02734004,A Phase I/II Study of MEDI4736 in Combination ...,"Ovarian, Breast, SCLC, Gastric Cancers","Olaparib (DRUG), MEDI4736 (DRUG), Bevacizumab ...",AstraZeneca,ACTIVE_NOT_RECRUITING,The purpose of this study is to look at the ef...,2016-03-17,2026-09-17,Drug,"[Olaparib, MEDI4736, Bevacizumab]",This Phase I/II trial evaluates MEDI4736 (durv...
1,NCT02264678,Ascending Doses of Ceralasertib in Combination...,"Adv Solid Malig - H&N SCC, ATM Pro / Def NSCLC...","Administration of ceralasertib (DRUG), Adminis...",AstraZeneca,ACTIVE_NOT_RECRUITING,"This is a modular, phase I/ phase 1 b, open-la...",2014-10-31,2025-12-31,Drug,"[ceralasertib, olaparib, durvalumab, AZD5305, ...",This Phase I/Ib trial evaluates ceralasertib (...
2,NCT06103864,A Phase III Study of Dato-DXd With or Without ...,Breast Cancer,"Dato-DXd (DRUG), Durvalumab (DRUG), Paclitaxel...",AstraZeneca,RECRUITING,"This is a Phase III, randomised, open-label, 3...",2023-11-23,2029-04-30,Drug,"[Dato-DXd, Durvalumab, Paclitaxel, Nab-paclita...","This Phase III trial (recruiting, started Nove..."
3,NCT02586987,"A Study to Assess the Safety, Tolerability and...","Lung Cancer, Melanoma, Head and Neck Carcinoma...","Selumetinib (DRUG), MEDI4736 (DRUG), Tremelimu...",AstraZeneca,COMPLETED,"This is a Phase I, open-label, multi-centre, d...",2015-12-28,2019-09-20,Drug,"[Selumetinib, MEDI4736, Tremelimumab]",This completed Phase I trial (2015-2019) evalu...
4,NCT04556773,A Phase 1b Study of T-DXd Combinations in HER2...,Metastatic Breast Cancer,"Trastuzumab deruxtecan (DRUG), Durvalumab (DRU...",AstraZeneca,ACTIVE_NOT_RECRUITING,"DESTINY-Breast 08 will investigate the safety,...",2020-12-17,2026-06-16,Drug,"[Trastuzumab deruxtecan, Durvalumab, Paclitaxe...",This Phase 1b trial (DESTINY-Breast 08) evalua...



Summary:
- Status distribution: {'ACTIVE_NOT_RECRUITING': 4, 'COMPLETED': 2, 'RECRUITING': 1}
- Intervention categories: {'Drug': 7}

Data saved to 'clinical_trials.csv'
